## Exercise — Where a Straight Line Goes Wrong

### Task 1 — Read the residuals of the alkane fit

Copy this setup into your session. It holds the alkane table, the line model and the R-squared helper from the video:

```py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

alkanes = pd.DataFrame({
    "name": ["methane", "ethane", "propane", "butane", "pentane",
             "hexane", "heptane", "octane", "nonane", "decane"],
    "n_carbons": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "bp_C": [-161.5, -88.6, -42.1, -0.5, 36.1, 68.7, 98.4, 125.6, 150.8, 174.1],
})

def line(x, slope, intercept):
    return slope * x + intercept

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot
```

Then do the following:

1. Fit `line` to boiling point against `n_carbons` with `curve_fit`.
2. Compute the residuals as measured − predicted.
3. Print each alkane's name next to its residual, rounded to 1 decimal place.
4. Draw the residual plot: residual against number of carbons, with a grey line at zero, axis labels with units, a title and a grid.

Before you move on, look at the printout and the plot. Note where the residuals are negative and where they are positive. Then decide what that pattern says about using a straight line for this data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

alkanes = pd.DataFrame({
    "name": ["methane", "ethane", "propane", "butane", "pentane",
             "hexane", "heptane", "octane", "nonane", "decane"],
    "n_carbons": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "bp_C": [-161.5, -88.6, -42.1, -0.5, 36.1, 68.7, 98.4, 125.6, 150.8, 174.1],
})

def line(x, slope, intercept):
    return slope * x + intercept

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

popt, pcov = curve_fit(line, alkanes["n_carbons"], alkanes["bp_C"])
slope, intercept = popt
bp_pred = line(alkanes["n_carbons"], slope, intercept)
residuals = alkanes["bp_C"] - bp_pred
for name, res in zip(alkanes["name"], residuals):
    print(name, round(res, 1))

plt.figure(figsize=(8, 5))
plt.scatter(alkanes["n_carbons"], residuals)
plt.plot([1, 10], [0, 0], color="grey")
plt.xlabel("Number of carbon atoms")
plt.ylabel("Residual (°C)")
plt.title("Residuals of the straight-line fit to alkane boiling points")
plt.grid(True)
plt.show()

### Task 2 — A decimal slip in the other direction

This time the octanol logP was typed as `0.30` instead of `3.00`. The other seven alcohols are correct:

| n_carbons | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| logP entered | −0.77 | −0.31 | 0.25 | 0.88 | 1.51 | 2.03 | 2.62 | 0.30 |

1. Before running anything, compare the octanol value with the trend of the other seven, about +0.56 per CH2. Decide whether 0.30 is plausible chemistry or an error.
2. Write down whether the fitted slope will come out larger or smaller than 0.56.
3. Build a table `alcohols_slip` with the columns `n_carbons`, `logp_entered` and `logp_checked`. In `logp_checked`, give octanol the source value 3.00.
4. Fit `line` to each logP column and print both slopes, rounded to 2 decimal places.


In [ ]:
alcohols_slip = pd.DataFrame({
    "n_carbons": [1, 2, 3, 4, 5, 6, 7, 8],
    "logp_entered": [-0.77, -0.31, 0.25, 0.88, 1.51, 2.03, 2.62, 0.30],
    "logp_checked": [-0.77, -0.31, 0.25, 0.88, 1.51, 2.03, 2.62, 3.00],
})
popt_bad, pcov_bad = curve_fit(line, alcohols_slip["n_carbons"], alcohols_slip["logp_entered"])
slope_bad, intercept_bad = popt_bad
popt_good, pcov_good = curve_fit(line, alcohols_slip["n_carbons"], alcohols_slip["logp_checked"])
slope_good, intercept_good = popt_good
print("slope with the slip (logP per carbon):", round(slope_bad, 2))
print("slope after checking (logP per carbon):", round(slope_good, 2))

### Task 3 — Inside or outside the range?

The alkane line from Task 1 was trained on 1 to 10 carbons. Look at these lines:

```py
print("hexane, predicted (°C):", round(line(6, slope, intercept)))
print("hexane, measured (°C):", 68.7)
print("dodecane, predicted (°C):", round(line(12, slope, intercept)))
print("dodecane, measured (°C):", 216.3)
```

Before you run them, write down two things:

- Which of the two predictions is interpolation, and which is extrapolation?
- Which prediction do you expect to be further from the measured value?

Then run the four lines and compare their output with what you wrote.


In [ ]:
print("hexane, predicted (°C):", round(line(6, slope, intercept)))
print("hexane, measured (°C):", 68.7)
print("dodecane, predicted (°C):", round(line(12, slope, intercept)))
print("dodecane, measured (°C):", 216.3)

### Task 4 — A high score from a cloud

Here are six compounds with their logP and pIC50:

```py
potency = pd.DataFrame({
    "logp": [1.0, 1.2, 1.4, 1.6, 1.8, 6.0],
    "pic50": [5.2, 4.8, 5.1, 4.9, 5.0, 8.0],
})
```

The first five compounds show no trend at all. The sixth sits far away from them.

1. Fit `line` to pIC50 against logP using all six compounds. Print R-squared, rounded to 2 decimal places.
2. Fit again using only the first five rows, `potency.iloc[:5]`. Print that R-squared too, rounded to 2 decimal places.

Before you run anything, predict roughly what each R-squared will be. Then compare the two printed values.


In [ ]:
potency = pd.DataFrame({
    "logp": [1.0, 1.2, 1.4, 1.6, 1.8, 6.0],
    "pic50": [5.2, 4.8, 5.1, 4.9, 5.0, 8.0],
})
popt_all, pcov_all = curve_fit(line, potency["logp"], potency["pic50"])
slope_all, intercept_all = popt_all
pic50_pred_all = line(potency["logp"], slope_all, intercept_all)
print("R-squared, all six:", round(r_squared(potency["pic50"], pic50_pred_all), 2))
cloud = potency.iloc[:5]
popt_cloud, pcov_cloud = curve_fit(line, cloud["logp"], cloud["pic50"])
slope_cloud, intercept_cloud = popt_cloud
pic50_pred_cloud = line(cloud["logp"], slope_cloud, intercept_cloud)
print("R-squared, first five:", round(r_squared(cloud["pic50"], pic50_pred_cloud), 2))